# Apache Iceberg — cenário de pedidos

## Domínio (fonte de dados)

Mesmo caso do notebook Delta: pedidos com **id**, **cliente** e **total**. Dados sintéticos no SQL abaixo; em produção poderiam vir de CSV/Parquet/API.

## Modelo ER (conceitual)

- **Pedido**: `id` (PK), `cliente`, `total`.

```mermaid
erDiagram
    PEDIDO {
        int id PK
        string cliente
        double total
    }
```

*(O PDF pede também imagens do ER: você pode exportar um diagrama e colocar em `docs/assets/`.)*

## DDL

```sql
CREATE TABLE local.pedidos_iceberg (
  id INT,
  cliente STRING,
  total DOUBLE
) USING iceberg;
```

Catálogo **`local`**, *warehouse* em `spark-warehouse/iceberg`. Em seguida: **INSERT**, **UPDATE** e **DELETE** via Spark SQL.


In [1]:
import os
from pathlib import Path

from pyspark.sql import SparkSession

# Windows: mesmo HADOOP_HOME que no notebook Delta (winutils + hadoop.dll).
_root = Path.cwd().resolve()
if not (_root / "tools" / "hadoop" / "bin" / "winutils.exe").exists():
    _root = _root.parent
_hadoop = _root / "tools" / "hadoop"
if _hadoop.joinpath("bin/winutils.exe").exists():
    os.environ["HADOOP_HOME"] = str(_hadoop)
    _bin = str(_hadoop / "bin")
    os.environ["PATH"] = _bin + os.pathsep + os.environ.get("PATH", "")

wh = Path("spark-warehouse/iceberg").resolve()
wh.mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder.appName("trabalho-iceberg").master("local[*]")
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.local.type", "hadoop")
    .config("spark.sql.catalog.local.warehouse", str(wh))
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")


In [2]:
spark.sql("DROP TABLE IF EXISTS local.pedidos_iceberg")
spark.sql("""
CREATE TABLE local.pedidos_iceberg (
  id INT,
  cliente STRING,
  total DOUBLE
) USING iceberg
""")


DataFrame[]

In [3]:
spark.sql("""
INSERT INTO local.pedidos_iceberg VALUES
  (1, 'Ana', 100.0),
  (2, 'Beto', 200.0),
  (3, 'Carla', 50.0)
""")
spark.sql("SELECT * FROM local.pedidos_iceberg ORDER BY id").show()


+---+-------+-----+
| id|cliente|total|
+---+-------+-----+
|  1|    Ana|100.0|
|  2|   Beto|200.0|
|  3|  Carla| 50.0|
+---+-------+-----+



In [4]:
spark.sql("UPDATE local.pedidos_iceberg SET total = 120.0 WHERE id = 1")
spark.sql("SELECT * FROM local.pedidos_iceberg ORDER BY id").show()


+---+-------+-----+
| id|cliente|total|
+---+-------+-----+
|  1|    Ana|120.0|
|  2|   Beto|200.0|
|  3|  Carla| 50.0|
+---+-------+-----+



In [5]:
spark.sql("DELETE FROM local.pedidos_iceberg WHERE id = 2")
spark.sql("SELECT * FROM local.pedidos_iceberg ORDER BY id").show()


+---+-------+-----+
| id|cliente|total|
+---+-------+-----+
|  1|    Ana|120.0|
|  3|  Carla| 50.0|
+---+-------+-----+

